In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# install Hangul font

!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Ign:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1
Ign:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 2min 58s (58.1 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf

In [25]:
import pandas as pd
import plotly.express as px

# 1. 데이터 불러오기
file_name = '/content/A0067885_최신데이터.xlsx'
df = pd.read_excel(file_name)

# 2. 데이터 전처리
df_firm = df[df['Firm/Forecast'].astype(str).str.strip().str.capitalize() == 'Firm'].copy()

# [정렬 준비] 월 이름을 숫자로 변환
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_firm['Month_Num'] = df_firm['Month'].str.strip().map(month_map)

# 3. 실제 날짜(Sorting_Date) 컬럼 생성 및 정렬
# Year, Month_Num, Day를 합쳐 진짜 '날짜' 데이터를 만듭니다.
df_firm['Sorting_Date'] = pd.to_datetime(pd.DataFrame({
    'year': df_firm['Year'],
    'month': df_firm['Month_Num'],
    'day': df_firm['Day']
}))

# [핵심] 만든 날짜를 기준으로 데이터를 완벽하게 정렬합니다.
df_firm = df_firm.sort_values(by='Sorting_Date')

# 4. 이상치(Outlier) 판별 및 그룹화
Q1 = df_firm['Quantity'].quantile(0.25)
Q3 = df_firm['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_firm['Display_Group'] = df_firm.apply(
    lambda x: 'Outlier (이상치)' if (x['Quantity'] < lower_bound or x['Quantity'] > upper_bound)
    else x['Month'], axis=1
)

# 5. 시각화 설정
# [요청사항] 이상치는 검은색(#000000), 나머지는 자동 색상
color_map = {'Outlier (이상치)': '#000000'}

# [핵심] 정렬된 데이터프레임에서 ID_Release의 순서를 리스트로 추출합니다.
# 이 리스트 순서가 곧 X축의 순서가 됩니다.
ordered_id_list = df_firm['ID_Release'].unique().tolist()

fig = px.scatter(
    df_firm,
    x='ID_Release',
    y='Quantity',
    color='Display_Group',
    color_discrete_map=color_map,
    hover_data={
        'ID_Release': True,
        'Year': True,
        'Month': True,
        'Day': True,
        'Quantity': ':.3f'
    },
    title='A0067885 ID_Release별 Firm Quantity Trend',
    labels={'Quantity': '수량 (pc)', 'ID_Release': '릴리즈 번호', 'Display_Group': '구분'},
    template='plotly_white'
)

# 6. X축 정렬 강제 고정 (이 부분이 핵심입니다)
fig.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=ordered_id_list, # 위에서 만든 날짜순 리스트를 주입
    tickangle=-45
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)

fig.update_layout(
    height=700,
    margin=dict(l=50, r=50, b=150, t=80),
    legend_title_text='월별 및 이상치',
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()

In [26]:
import pandas as pd
import plotly.express as px

# 1. 데이터 불러오기
file_name = '/content/A0028219_최신데이터.xlsx'
df = pd.read_excel(file_name)

# 2. 데이터 전처리
df_firm = df[df['Firm/Forecast'].astype(str).str.strip().str.capitalize() == 'Firm'].copy()

# [정렬 준비] 월 이름을 숫자로 변환
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_firm['Month_Num'] = df_firm['Month'].str.strip().map(month_map)

# 3. 실제 날짜(Sorting_Date) 컬럼 생성 및 정렬
# Year, Month_Num, Day를 합쳐 진짜 '날짜' 데이터를 만듭니다.
df_firm['Sorting_Date'] = pd.to_datetime(pd.DataFrame({
    'year': df_firm['Year'],
    'month': df_firm['Month_Num'],
    'day': df_firm['Day']
}))

# [핵심] 만든 날짜를 기준으로 데이터를 완벽하게 정렬합니다.
df_firm = df_firm.sort_values(by='Sorting_Date')

# 4. 이상치(Outlier) 판별 및 그룹화
Q1 = df_firm['Quantity'].quantile(0.25)
Q3 = df_firm['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_firm['Display_Group'] = df_firm.apply(
    lambda x: 'Outlier (이상치)' if (x['Quantity'] < lower_bound or x['Quantity'] > upper_bound)
    else x['Month'], axis=1
)

# 5. 시각화 설정
# [요청사항] 이상치는 검은색(#000000), 나머지는 자동 색상
color_map = {'Outlier (이상치)': '#000000'}

# [핵심] 정렬된 데이터프레임에서 ID_Release의 순서를 리스트로 추출합니다.
# 이 리스트 순서가 곧 X축의 순서가 됩니다.
ordered_id_list = df_firm['ID_Release'].unique().tolist()

fig = px.scatter(
    df_firm,
    x='ID_Release',
    y='Quantity',
    color='Display_Group',
    color_discrete_map=color_map,
    hover_data={
        'ID_Release': True,
        'Year': True,
        'Month': True,
        'Day': True,
        'Quantity': ':.3f'
    },
    title='A0028219 ID_Release별 Firm Quantity Trend',
    labels={'Quantity': '수량 (pc)', 'ID_Release': '릴리즈 번호', 'Display_Group': '구분'},
    template='plotly_white'
)

# 6. X축 정렬 강제 고정 (이 부분이 핵심입니다)
fig.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=ordered_id_list, # 위에서 만든 날짜순 리스트를 주입
    tickangle=-45
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)

fig.update_layout(
    height=700,
    margin=dict(l=50, r=50, b=150, t=80),
    legend_title_text='월별 및 이상치',
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()

In [27]:
import pandas as pd
import plotly.express as px

# 1. 데이터 불러오기
file_name = '/content/A0030814X_최신데이터.xlsx'
df = pd.read_excel(file_name)

# 2. 데이터 전처리
df_firm = df[df['Firm/Forecast'].astype(str).str.strip().str.capitalize() == 'Firm'].copy()

# [정렬 준비] 월 이름을 숫자로 변환
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_firm['Month_Num'] = df_firm['Month'].str.strip().map(month_map)

# 3. 실제 날짜(Sorting_Date) 컬럼 생성 및 정렬
# Year, Month_Num, Day를 합쳐 진짜 '날짜' 데이터를 만듭니다.
df_firm['Sorting_Date'] = pd.to_datetime(pd.DataFrame({
    'year': df_firm['Year'],
    'month': df_firm['Month_Num'],
    'day': df_firm['Day']
}))

# [핵심] 만든 날짜를 기준으로 데이터를 완벽하게 정렬합니다.
df_firm = df_firm.sort_values(by='Sorting_Date')

# 4. 이상치(Outlier) 판별 및 그룹화
Q1 = df_firm['Quantity'].quantile(0.25)
Q3 = df_firm['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_firm['Display_Group'] = df_firm.apply(
    lambda x: 'Outlier (이상치)' if (x['Quantity'] < lower_bound or x['Quantity'] > upper_bound)
    else x['Month'], axis=1
)

# 5. 시각화 설정
# [요청사항] 이상치는 검은색(#000000), 나머지는 자동 색상
color_map = {'Outlier (이상치)': '#000000'}

# [핵심] 정렬된 데이터프레임에서 ID_Release의 순서를 리스트로 추출합니다.
# 이 리스트 순서가 곧 X축의 순서가 됩니다.
ordered_id_list = df_firm['ID_Release'].unique().tolist()

fig = px.scatter(
    df_firm,
    x='ID_Release',
    y='Quantity',
    color='Display_Group',
    color_discrete_map=color_map,
    hover_data={
        'ID_Release': True,
        'Year': True,
        'Month': True,
        'Day': True,
        'Quantity': ':.3f'
    },
    title='A0030814X ID_Release별 Firm Quantity Trend',
    labels={'Quantity': '수량 (pc)', 'ID_Release': '릴리즈 번호', 'Display_Group': '구분'},
    template='plotly_white'
)

# 6. X축 정렬 강제 고정 (이 부분이 핵심입니다)
fig.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=ordered_id_list, # 위에서 만든 날짜순 리스트를 주입
    tickangle=-45
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)

fig.update_layout(
    height=700,
    margin=dict(l=50, r=50, b=150, t=80),
    legend_title_text='월별 및 이상치',
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()

In [28]:
import pandas as pd
import plotly.express as px

# 1. 데이터 불러오기
file_name = '/content/A0049000_최신데이터.xlsx'
df = pd.read_excel(file_name)

# 2. 데이터 전처리
df_firm = df[df['Firm/Forecast'].astype(str).str.strip().str.capitalize() == 'Firm'].copy()

# [정렬 준비] 월 이름을 숫자로 변환
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_firm['Month_Num'] = df_firm['Month'].str.strip().map(month_map)

# 3. 실제 날짜(Sorting_Date) 컬럼 생성 및 정렬
# Year, Month_Num, Day를 합쳐 진짜 '날짜' 데이터를 만듭니다.
df_firm['Sorting_Date'] = pd.to_datetime(pd.DataFrame({
    'year': df_firm['Year'],
    'month': df_firm['Month_Num'],
    'day': df_firm['Day']
}))

# [핵심] 만든 날짜를 기준으로 데이터를 완벽하게 정렬합니다.
df_firm = df_firm.sort_values(by='Sorting_Date')

# 4. 이상치(Outlier) 판별 및 그룹화
Q1 = df_firm['Quantity'].quantile(0.25)
Q3 = df_firm['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_firm['Display_Group'] = df_firm.apply(
    lambda x: 'Outlier (이상치)' if (x['Quantity'] < lower_bound or x['Quantity'] > upper_bound)
    else x['Month'], axis=1
)

# 5. 시각화 설정
# [요청사항] 이상치는 검은색(#000000), 나머지는 자동 색상
color_map = {'Outlier (이상치)': '#000000'}

# [핵심] 정렬된 데이터프레임에서 ID_Release의 순서를 리스트로 추출합니다.
# 이 리스트 순서가 곧 X축의 순서가 됩니다.
ordered_id_list = df_firm['ID_Release'].unique().tolist()

fig = px.scatter(
    df_firm,
    x='ID_Release',
    y='Quantity',
    color='Display_Group',
    color_discrete_map=color_map,
    hover_data={
        'ID_Release': True,
        'Year': True,
        'Month': True,
        'Day': True,
        'Quantity': ':.3f'
    },
    title='A0049000 ID_Release별 Firm Quantity Trend',
    labels={'Quantity': '수량 (pc)', 'ID_Release': '릴리즈 번호', 'Display_Group': '구분'},
    template='plotly_white'
)

# 6. X축 정렬 강제 고정 (이 부분이 핵심입니다)
fig.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=ordered_id_list, # 위에서 만든 날짜순 리스트를 주입
    tickangle=-45
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)

fig.update_layout(
    height=700,
    margin=dict(l=50, r=50, b=150, t=80),
    legend_title_text='월별 및 이상치',
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()

In [29]:
import pandas as pd
import plotly.express as px

# 1. 데이터 불러오기
file_name = '/content/A0070270_최신데이터.xlsx'
df = pd.read_excel(file_name)

# 2. 데이터 전처리
df_firm = df[df['Firm/Forecast'].astype(str).str.strip().str.capitalize() == 'Firm'].copy()

# [정렬 준비] 월 이름을 숫자로 변환
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_firm['Month_Num'] = df_firm['Month'].str.strip().map(month_map)

# 3. 실제 날짜(Sorting_Date) 컬럼 생성 및 정렬
# Year, Month_Num, Day를 합쳐 진짜 '날짜' 데이터를 만듭니다.
df_firm['Sorting_Date'] = pd.to_datetime(pd.DataFrame({
    'year': df_firm['Year'],
    'month': df_firm['Month_Num'],
    'day': df_firm['Day']
}))

# [핵심] 만든 날짜를 기준으로 데이터를 완벽하게 정렬합니다.
df_firm = df_firm.sort_values(by='Sorting_Date')

# 4. 이상치(Outlier) 판별 및 그룹화
Q1 = df_firm['Quantity'].quantile(0.25)
Q3 = df_firm['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_firm['Display_Group'] = df_firm.apply(
    lambda x: 'Outlier (이상치)' if (x['Quantity'] < lower_bound or x['Quantity'] > upper_bound)
    else x['Month'], axis=1
)

# 5. 시각화 설정
# [요청사항] 이상치는 검은색(#000000), 나머지는 자동 색상
color_map = {'Outlier (이상치)': '#000000'}

# [핵심] 정렬된 데이터프레임에서 ID_Release의 순서를 리스트로 추출합니다.
# 이 리스트 순서가 곧 X축의 순서가 됩니다.
ordered_id_list = df_firm['ID_Release'].unique().tolist()

fig = px.scatter(
    df_firm,
    x='ID_Release',
    y='Quantity',
    color='Display_Group',
    color_discrete_map=color_map,
    hover_data={
        'ID_Release': True,
        'Year': True,
        'Month': True,
        'Day': True,
        'Quantity': ':.3f'
    },
    title='A0070270 ID_Release별 Firm Quantity Trend',
    labels={'Quantity': '수량 (pc)', 'ID_Release': '릴리즈 번호', 'Display_Group': '구분'},
    template='plotly_white'
)

# 6. X축 정렬 강제 고정 (이 부분이 핵심입니다)
fig.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=ordered_id_list, # 위에서 만든 날짜순 리스트를 주입
    tickangle=-45
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)

fig.update_layout(
    height=700,
    margin=dict(l=50, r=50, b=150, t=80),
    legend_title_text='월별 및 이상치',
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()

In [30]:
import pandas as pd
import plotly.express as px

# 1. 데이터 불러오기
file_name = '/content/A021N565_최신데이터.xlsx'
df = pd.read_excel(file_name)

# 2. 데이터 전처리
df_firm = df[df['Firm/Forecast'].astype(str).str.strip().str.capitalize() == 'Firm'].copy()

# [정렬 준비] 월 이름을 숫자로 변환
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_firm['Month_Num'] = df_firm['Month'].str.strip().map(month_map)

# 3. 실제 날짜(Sorting_Date) 컬럼 생성 및 정렬
# Year, Month_Num, Day를 합쳐 진짜 '날짜' 데이터를 만듭니다.
df_firm['Sorting_Date'] = pd.to_datetime(pd.DataFrame({
    'year': df_firm['Year'],
    'month': df_firm['Month_Num'],
    'day': df_firm['Day']
}))

# [핵심] 만든 날짜를 기준으로 데이터를 완벽하게 정렬합니다.
df_firm = df_firm.sort_values(by='Sorting_Date')

# 4. 이상치(Outlier) 판별 및 그룹화
Q1 = df_firm['Quantity'].quantile(0.25)
Q3 = df_firm['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_firm['Display_Group'] = df_firm.apply(
    lambda x: 'Outlier (이상치)' if (x['Quantity'] < lower_bound or x['Quantity'] > upper_bound)
    else x['Month'], axis=1
)

# 5. 시각화 설정
# [요청사항] 이상치는 검은색(#000000), 나머지는 자동 색상
color_map = {'Outlier (이상치)': '#000000'}

# [핵심] 정렬된 데이터프레임에서 ID_Release의 순서를 리스트로 추출합니다.
# 이 리스트 순서가 곧 X축의 순서가 됩니다.
ordered_id_list = df_firm['ID_Release'].unique().tolist()

fig = px.scatter(
    df_firm,
    x='ID_Release',
    y='Quantity',
    color='Display_Group',
    color_discrete_map=color_map,
    hover_data={
        'ID_Release': True,
        'Year': True,
        'Month': True,
        'Day': True,
        'Quantity': ':.3f'
    },
    title='A0021N565 ID_Release별 Firm Quantity Trend',
    labels={'Quantity': '수량 (pc)', 'ID_Release': '릴리즈 번호', 'Display_Group': '구분'},
    template='plotly_white'
)

# 6. X축 정렬 강제 고정 (이 부분이 핵심입니다)
fig.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=ordered_id_list, # 위에서 만든 날짜순 리스트를 주입
    tickangle=-45
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)

fig.update_layout(
    height=700,
    margin=dict(l=50, r=50, b=150, t=80),
    legend_title_text='월별 및 이상치',
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()

**이상치 판단 방법**

In [ ]:
#데이터를 수량 순서대로 줄 세웠을 때, 다음과 같은 지표를 계산함.
#Q_1 (제1사분위수): 하위 25% 지점의 값
#Q_3 (제3사분위수): 상위 25% 지점(전체에서 75%)의 값
#IQR (사분위수 범위): Q_3 - Q_1 (가운데 50% 데이터가 모여 있는 구간의 길이)

#이 IQR 값을 바탕으로 정상 범위를 결정하는 상한선과 하한선을 만듦.
#상한선 (Upper Bound): Q_3 + (1.5 * times IQR)
#하한선 (Lower Bound): Q_1 - (1.5 * times IQR)
#이 계산 결과보다 더 크거나 더 작은 값이 들어오면, 코드에서 자동으로 이를 Outlier(이상치)로 분류하고 검은색으로 표시하게 됨.